# Parameter Golf - Google Colab

OpenAI Model Craft Challenge: Parameter Golf

**ランタイム設定**: 「ランタイム」→「ランタイムのタイプを変更」→ **T4 GPU** を選択

In [ ]:
!nvidia-smi

## 1. セットアップ

In [ ]:
import os, shutil
if os.path.exists('/content/parameter-golf'):
    shutil.rmtree('/content/parameter-golf')
!git clone https://github.com/tsubasagit/parameter-golf.git /content/parameter-golf
%cd /content/parameter-golf
!pip install -q sentencepiece huggingface-hub datasets tqdm zstandard
!python3 data/cached_challenge_fineweb.py --variant sp1024 --train-shards 1

## 2. T4 GPUパッチ適用

T4は bfloat16/Flash Attention/enable_gqa を非サポート。全スクリプトにパッチ適用。

In [ ]:
import glob

def patch_t4(path):
    """T4互換パッチ: bfloat16→float16, flash→math SDP, enable_gqa→repeat_interleave"""
    with open(path) as f:
        lines = f.readlines()

    out = []
    i = 0
    changed = False
    while i < len(lines):
        line = lines[i]

        # --- Fix 1: bfloat16 → float16 ---
        if 'torch.bfloat16' in line and 'orig_dtype' not in line:
            line = line.replace('torch.bfloat16', 'torch.float16')
            changed = True
        if '.bfloat16()' in line:
            line = line.replace('.bfloat16()', '.half()')
            changed = True

        # --- Fix 2: Flash SDP → math+mem_efficient SDP ---
        if 'enable_flash_sdp(True)' in line:
            line = line.replace('enable_flash_sdp(True)', 'enable_flash_sdp(False)')
            changed = True
        if 'enable_mem_efficient_sdp(False)' in line:
            line = line.replace('enable_mem_efficient_sdp(False)', 'enable_mem_efficient_sdp(True)')
            changed = True
        if 'enable_math_sdp(False)' in line:
            line = line.replace('enable_math_sdp(False)', 'enable_math_sdp(True)')
            changed = True

        # --- Fix 3: enable_gqa → repeat_interleave ---
        if 'enable_gqa=' in line:
            # Skip this line (remove enable_gqa argument)
            # Find the SDPA call start and inject repeat_interleave before it
            # Look backwards for the 'y = F.scaled_dot_product_attention(' line
            sdpa_idx = None
            for j in range(len(out)-1, max(len(out)-10, -1), -1):
                if 'F.scaled_dot_product_attention(' in out[j]:
                    sdpa_idx = j
                    break
            if sdpa_idx is not None:
                # Get indentation from the SDPA line
                indent = '        '
                repeat_code = (
                    f'{indent}if self.num_kv_heads != self.num_heads:\n'
                    f'{indent}    rep = self.num_heads // self.num_kv_heads\n'
                    f'{indent}    k = k.repeat_interleave(rep, dim=1)\n'
                    f'{indent}    v = v.repeat_interleave(rep, dim=1)\n'
                )
                out.insert(sdpa_idx, repeat_code)
            changed = True
            i += 1
            continue

        out.append(line)
        i += 1

    if changed:
        with open(path, 'w') as f:
            f.writelines(out)
        print(f'  Patched: {path}')
    else:
        print(f'  No changes: {path}')

# 全train_gpt*.pyにパッチ
targets = glob.glob('train_gpt*.py') + glob.glob('records/**/train_gpt*.py', recursive=True)
print(f'Patching {len(targets)} files...')
for t in targets:
    patch_t4(t)

# 検証
import subprocess
for check, label in [('enable_gqa', 'enable_gqa'), ('enable_flash_sdp(True)', 'flash_sdp'), ('torch.bfloat16', 'bfloat16')]:
    r = subprocess.run(['grep', '-rn', check, 'train_gpt.py'], capture_output=True, text=True)
    status = 'WARN: still found' if r.stdout.strip() else 'OK: removed'
    print(f'{label}: {status}')

---
## 3. ベースライン実行

In [ ]:
import os
os.environ['RUN_ID'] = 'colab_baseline'
os.environ['ITERATIONS'] = '500'
os.environ['TRAIN_BATCH_TOKENS'] = '131072'
os.environ['VAL_LOSS_EVERY'] = '100'
os.environ['VAL_BATCH_SIZE'] = '65536'
os.environ['MAX_WALLCLOCK_SECONDS'] = '600'

!torchrun --standalone --nproc_per_node=1 train_gpt.py

---
## 4. 改良版: 上位テクニック適用

| テクニック | 効果 |
|-----------|------|
| MLP 3x拡張 | hidden dim 1024→1536 |
| SmearGate | 前トークンとのゲート融合 |
| BigramHash(4096) | トークンペアのハッシュ埋め込み |
| U-Net Skip | スキップ接続 |
| SWA | チェックポイント平均 |
| Muon WD | Weight Decay |
| Sliding Window Eval | stride=64 |

In [ ]:
import shutil
src = '/content/parameter-golf/records/track_10min_16mb/2026-03-20_Int6_MLP3x_SmearGate_BigramHash_MuonWD_SWA/train_gpt.py'
dst = '/content/parameter-golf/train_gpt_improved.py'
shutil.copy2(src, dst)
print('Copied improved script (already T4-patched)')

In [ ]:
import os

os.environ['RUN_ID'] = 'colab_improved_v1'
os.environ['DATA_PATH'] = './data/datasets/fineweb10B_sp1024'
os.environ['TOKENIZER_PATH'] = './data/tokenizers/fineweb_1024_bpe.model'
os.environ['VOCAB_SIZE'] = '1024'
os.environ['ITERATIONS'] = '500'
os.environ['MAX_WALLCLOCK_SECONDS'] = '900'
os.environ['TRAIN_BATCH_TOKENS'] = '65536'
os.environ['VAL_BATCH_SIZE'] = '65536'
os.environ['TRAIN_SEQ_LEN'] = '1024'
os.environ['VAL_LOSS_EVERY'] = '100'
os.environ['NUM_LAYERS'] = '9'
os.environ['MODEL_DIM'] = '512'
os.environ['NUM_HEADS'] = '8'
os.environ['NUM_KV_HEADS'] = '4'
os.environ['MLP_MULT'] = '3'
os.environ['MATRIX_LR'] = '0.02'
os.environ['SCALAR_LR'] = '0.02'
os.environ['MUON_MOMENTUM'] = '0.99'
os.environ['WARMDOWN_ITERS'] = '200'

!torchrun --standalone --nproc_per_node=1 train_gpt_improved.py

---
## 5. カスタム実験

In [ ]:
import os

os.environ['RUN_ID'] = 'colab_experiment_custom'
os.environ['DATA_PATH'] = './data/datasets/fineweb10B_sp1024'
os.environ['TOKENIZER_PATH'] = './data/tokenizers/fineweb_1024_bpe.model'
os.environ['VOCAB_SIZE'] = '1024'
os.environ['ITERATIONS'] = '500'
os.environ['MAX_WALLCLOCK_SECONDS'] = '900'
os.environ['TRAIN_BATCH_TOKENS'] = '65536'
os.environ['VAL_BATCH_SIZE'] = '65536'
os.environ['VAL_LOSS_EVERY'] = '100'

# --- ここを変更 ---
os.environ['NUM_LAYERS'] = '10'
os.environ['TRAIN_SEQ_LEN'] = '1024'
os.environ['MLP_MULT'] = '3'
os.environ['MATRIX_LR'] = '0.02'
os.environ['MUON_MOMENTUM'] = '0.99'
os.environ['WARMDOWN_ITERS'] = '200'

!torchrun --standalone --nproc_per_node=1 train_gpt_improved.py